# ViewViewView 卡顿复现测试

**用法**：从上到下依次运行（Shift+Enter）。右上角会出现一个黑色监控条，实时显示：

- `FPS`：浏览器实际帧率。实验运行时如果掉到 10 以下并且长时间不恢复，就是渲染跟不上。
- `JS堆`：浏览器 JS 堆。持续上涨且不回落 = 消息积压或对象堆积。
- `DOM`：节点总数。正常应基本恒定（widget 自己约 6000 个）。

> 注意：**不要把浏览器窗口最小化或切到别的标签页**，后台标签页的 rAF 会被限到 1fps，数据会失真。

---

## 判定口径

| 现象 | 含义 |
| --- | --- |
| FPS 稳定在 15~20（`frame_delay=0.05`），堆平稳 | 正常 |
| 500 行时 FPS < 10 且堆持续涨 | 复现了你说的卡顿，请把三行数字发我 |
| 只有 `frame_delay=0.2` 才顺畅 | 渲染算力不足，需要 P1（增量行更新） |
| `LIVE_ROWS=200` 顺畅、1000 卡 | 深度太大，需要 P3（push_depth） |
| 注入本身很慢（第 2 个测试耗时远超预期） | kernel/磁盘问题（Defender 扫描 CSV 等），不是前端 |


In [1]:
import sys

import anywidget
import ipywidgets
import viewviewview
from ipywidgets import Widget

from ipywidgets import Widget


def live_widget_count():
    try:
        return len(Widget.widgets)  # ipywidgets 8 (deprecated but present)
    except Exception:
        try:
            return len(Widget._instances)  # older ipywidgets
        except Exception:
            return "n/a"


print("viewviewview:", viewviewview.__version__)
print("path        :", viewviewview.__file__)
print("anywidget   :", anywidget.__version__)
print("ipywidgets  :", ipywidgets.__version__)
print("python      :", sys.version.split()[0])
print("live widget models:", live_widget_count())


viewviewview: 0.2.0
path        : D:\python\Lib\site-packages\viewviewview\__init__.py
anywidget   : 0.11.0
ipywidgets  : 8.1.9
python      : 3.12.0
live widget models: 0


C:\Users\Kong\AppData\Local\Temp\ipykernel_21816\1526765170.py:13: DeprecationWarning: Widget.widgets is deprecated.
  return len(Widget.widgets)  # ipywidgets 8 (deprecated but present)


## 1. 参数（先按默认跑一遍，再按需改）

- `ROWS`：本次注入多少条
- `RATE`：每秒注入多少条；设 `0` 表示全速（不打间隔）
- `LIVE_ROWS`：环形窗口大小，也决定明细窗深度
- `FRAME_DELAY`：前端推送合帧间隔；`0.05` = 最多 20 帧/秒，`0.2` = 5 帧/秒


In [2]:
ROWS = 500          # 先跑 500，复现你说的场景
RATE = 100          # 每秒 100 条；0 = 全速
LIVE_ROWS = 1000
FRAME_DELAY = 0.05  # 卡的话改成 0.2 再跑一次对比

COLUMNS = {
    "state": str,
    "delay": int,
    "pulse": int,
    "reply": str,
    "errcode": int,
}


## 2. 启动浏览器监控条

In [12]:
from IPython.display import Javascript, display

display(
    Javascript(
        r"""(() => {
  if (window.__vvvMonitor) { window.__vvvMonitor.stop(); }
  const box = document.createElement('div');
  box.style.cssText = [
    'position:fixed', 'right:12px', 'bottom:12px', 'z-index:99999',
    'background:#111', 'color:#eee', 'font:12px/1.6 Menlo,Consolas,monospace',
    'padding:10px 12px', 'border-radius:10px',
    'box-shadow:0 4px 20px rgba(0,0,0,.45)', 'white-space:pre', 'text-align:left'
  ].join(';');
  document.body.appendChild(box);
  let frames = 0;
  let last = performance.now();
  let fps = 0;
  const tick = () => { frames += 1; requestAnimationFrame(tick); };
  requestAnimationFrame(tick);
  const timer = setInterval(() => {
    const now = performance.now();
    fps = frames * 1000 / (now - last);
    frames = 0;
    last = now;
    const heap = performance.memory
      ? (performance.memory.usedJSHeapSize / 1048576).toFixed(1)
      : 'n/a';
    const nodes = document.getElementsByTagName('*').length;
    box.textContent = `FPS ${fps.toFixed(1)} | JS堆 ${heap} MB | DOM ${nodes} 节点`;
  }, 1000);
  window.__vvvMonitor = {
    stop() { clearInterval(timer); box.remove(); delete window.__vvvMonitor; }
  };
})()"""
    )
)


<IPython.core.display.Javascript object>

## 3. 创建并显示看板（`replace=True`，重跑 cell 只保留一个面板）

In [9]:
import random

import viewviewview as faultviz

vt = faultviz.ViewWidget(
    live_rows=LIVE_ROWS,
    frame_delay=FRAME_DELAY,
    flush_every=100,
    columns=COLUMNS,
)
vt.show(replace=True)
print("csv:", vt.csv_path)


2026-11-04 10:25:24,499 - viewviewview - INFO - No data recorded yet; showing the declared columns.


csv: D:\project\ViewViewView\dev\20261104-102524.csv


## 4. 全速突发 500 条

跑完盯着监控条看 3~5 秒：FPS 应该回到 15+，堆应该回落。


In [23]:
import time

t0 = time.perf_counter()
for i in range(ROWS):
    reply = random.choice(["ok", "no-reply", "short"])
    vt.update(
        state="PASS" if reply == "ok" else "FAIL",
        delay=random.randint(100, 2000),
        pulse=random.randint(10, 50),
        reply=reply,
        errcode=0 if reply == "ok" else random.randint(1, 8),
    )
elapsed = time.perf_counter() - t0
print(f"{ROWS} 条全速注入: {elapsed:.3f}s ({ROWS / elapsed:.0f} 行/秒), 写入 {vt.rows_written} 行")
print("现在观察监控条 3~5 秒：FPS 应回到 15+，堆应回落")


500 条全速注入: 0.005s (108495 行/秒), 写入 11868 行
现在观察监控条 3~5 秒：FPS 应回到 15+，堆应回落


## 5. 按固定速率持续注入（更接近真实实验）

这步会持续约 20 秒。观察过程中 FPS 和堆是否单调恶化。


In [24]:
interval = 1.0 / RATE if RATE else 0.0
start = time.perf_counter()
for i in range(ROWS * 2):
    reply = random.choice(["ok", "no-reply", "short"])
    vt.update(
        state="PASS" if reply == "ok" else "FAIL",
        delay=random.randint(100, 2000),
        pulse=random.randint(10, 50),
        reply=reply,
        errcode=0 if reply == "ok" else random.randint(1, 8),
    )
    if interval:
        time.sleep(interval)
print(
    f"按 {RATE or '全速'} 行/秒注入 {ROWS * 2} 条，"
    f"耗时 {time.perf_counter() - start:.1f}s，总计 {vt.rows_written} 行"
)


按 100 行/秒注入 1000 条，耗时 12.5s，总计 12868 行


## 6. 面板 / 内核侧体检

`live widget models` 如果远大于 1，说明之前重跑 cell 攒了旧面板（每个都在收推送、各自渲染）。


In [7]:
print("live widget models:", live_widget_count())
print("syncer attached panels:", vt.syncer.panel_count())
print("csv:", vt.csv_path)
print("rows written:", vt.rows_written)

# 面板累积演示：默认 show() 会再加一个面板；replace=True 只回收「上一次 show」
# 创建的那个面板，不会清理更早的面板（要彻底清零请重启 kernel 或清空输出）。
vt.show()
print("after plain show():", vt.syncer.panel_count())
vt.show(replace=True)
print("after replace=True:", vt.syncer.panel_count())


live widget models: 2
syncer attached panels: 1
csv: D:\project\ViewViewView\dev\20261104-102504.csv
rows written: 1500


C:\Users\Kong\AppData\Local\Temp\ipykernel_21816\1526765170.py:13: DeprecationWarning: Widget.widgets is deprecated.
  return len(Widget.widgets)  # ipywidgets 8 (deprecated but present)


after plain show(): 2


after replace=True: 2


## 7. 对照实验（按需）

卡的话，把第 1 节参数改成下面任一组，**重启 kernel 后重跑**，比较监控条：

| 对照组 | LIVE_ROWS | FRAME_DELAY | 说明 |
| --- | --- | --- | --- |
| A（默认） | 1000 | 0.05 | 基线 |
| B | 1000 | 0.2 | 只降刷新率 |
| C | 200 | 0.05 | 只降深度 |
| D | 200 | 0.2 | 两者都降 |

把 A/B/C/D 四组的 `FPS`、`JS堆`、以及第 4 步的注入耗时记下来发我，就能定位到底是渲染深度、刷新频率，还是 kernel/磁盘的锅。


## 8. 收尾

In [8]:
vt.close()
print("closed. syncer panels:", vt.syncer.panel_count())

# 停掉浏览器监控条（可选）
from IPython.display import Javascript, display

display(Javascript("window.__vvvMonitor && window.__vvvMonitor.stop()"))


closed. syncer panels: 0


<IPython.core.display.Javascript object>